In [4]:
import os
import time
from google import genai
from google.genai import types
from dotenv import load_dotenv
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


from datasets import load_dataset

In [5]:
ds = load_dataset("dair-ai/emotion", "split")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    2000 non-null   object
 1   label   2000 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 31.4+ KB


In [6]:
test['label'] = test['label'].map({0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'})

labels = test['label'].unique()

test

,text,label
0,im feeling rather rotten so im not very ambiti...,sadness
1,im updating my blog because i feel shitty,sadness
2,i never make her separate from me because i do...,sadness
3,i left with my bouquet of red and yellow tulip...,joy
4,i was feeling a little vain when i did this one,sadness
...,...,...
1995,i just keep feeling like someone is being unki...,anger
1996,im feeling a little cranky negative after this...,anger
1997,i feel that i am useful to my people and that ...,joy
1998,im feeling more comfortable with derby i feel ...,joy


In [7]:
load_dotenv()

api_key=os.environ.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

In [8]:
def classify(text, labels):

    sys_instruct="You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling multiclass classification tasks based on user instructions."

    start_time = time.time()

    response = client.models.generate_content(
        model="gemini-2.0-flash",
        config=types.GenerateContentConfig(
            system_instruction=sys_instruct,
            safety_settings=[
            types.SafetySetting(
                category="HARM_CATEGORY_HARASSMENT",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_HATE_SPEECH",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_SEXUALLY_EXPLICIT",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_DANGEROUS_CONTENT",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_CIVIC_INTEGRITY",
                threshold="BLOCK_NONE"
            ),
            ],
        ),
        contents=f"Classify the following text based on the task: Sentiment analysis of tweets. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"
    )

    request_time = time.time() - start_time
    completion = response.text
    if completion:
        completion = completion.lower()
    else:
        completion = "None"
    completion_tokens = response.usage_metadata.candidates_token_count
    prompt_tokens = response.usage_metadata.prompt_token_count
    total_tokens = response.usage_metadata.total_token_count

    return completion, request_time, completion_tokens, prompt_tokens, total_tokens

def post_process(text):
    if 'love' in text:
        return 'love'
    elif 'joy' in text:
        return 'joy'
    elif 'sadness' in text:
        return 'sadness'
    elif 'anger' in text:
        return 'anger'
    elif 'fear' in text:
        return 'fear'
    elif 'surprise' in text:
        return 'surprise'
    else:
        return 'error'

In [9]:
pred_df = test.copy() 

for index, row in pred_df.iterrows():
    try:
        text = row['text']
        completion, request_time, completion_tokens, prompt_tokens, total_tokens = classify(text, labels)
        pred_df.at[index, 'prediction'] = completion
        pred_df.at[index, 'request_time'] = request_time
        pred_df.at[index, 'completion_tokens'] = completion_tokens
        pred_df.at[index, 'prompt_tokens'] = prompt_tokens
        pred_df.at[index, 'total_tokens'] = total_tokens

    except Exception as e:
        # Save the current state of the DataFrame to a file before breaking out or retrying.
        pred_df.to_csv("results/partial_gemini_ZS_multiclass3.csv", index=False)
        print(f"An error occurred at index {index}: {e}. Partial results saved.")
        # Optionally, you can break out of the loop or continue based on your needs.
        break

pred_df['prediction_post_processed'] = pred_df['prediction'].apply(post_process)
pred_df.to_csv("results/gemini_ZS_multiclass3.csv", index=False)

pred_df

,text,label,prediction,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed
0,im feeling rather rotten so im not very ambiti...,sadness,sadness\n,1.185900,3.0,94.0,97.0,sadness
1,im updating my blog because i feel shitty,sadness,sadness\n,1.097142,3.0,91.0,94.0,sadness
2,i never make her separate from me because i do...,sadness,love\n,1.251199,2.0,105.0,107.0,love
3,i left with my bouquet of red and yellow tulip...,joy,joy\n,1.138867,2.0,104.0,106.0,joy
4,i was feeling a little vain when i did this one,sadness,joy\n,0.587485,2.0,94.0,96.0,joy
...,...,...,...,...,...,...,...,...
1995,i just keep feeling like someone is being unki...,anger,anger\n,1.126279,2.0,119.0,121.0,anger
1996,im feeling a little cranky negative after this...,anger,anger\n,0.566578,2.0,94.0,96.0,anger
1997,i feel that i am useful to my people and that ...,joy,joy\n,0.596335,2.0,101.0,103.0,joy
1998,im feeling more comfortable with derby i feel ...,joy,joy\n,1.047610,2.0,101.0,103.0,joy


In [10]:
pred_df

,text,label,prediction,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed
0,im feeling rather rotten so im not very ambiti...,sadness,sadness\n,1.185900,3.0,94.0,97.0,sadness
1,im updating my blog because i feel shitty,sadness,sadness\n,1.097142,3.0,91.0,94.0,sadness
2,i never make her separate from me because i do...,sadness,love\n,1.251199,2.0,105.0,107.0,love
3,i left with my bouquet of red and yellow tulip...,joy,joy\n,1.138867,2.0,104.0,106.0,joy
4,i was feeling a little vain when i did this one,sadness,joy\n,0.587485,2.0,94.0,96.0,joy
...,...,...,...,...,...,...,...,...
1995,i just keep feeling like someone is being unki...,anger,anger\n,1.126279,2.0,119.0,121.0,anger
1996,im feeling a little cranky negative after this...,anger,anger\n,0.566578,2.0,94.0,96.0,anger
1997,i feel that i am useful to my people and that ...,joy,joy\n,0.596335,2.0,101.0,103.0,joy
1998,im feeling more comfortable with derby i feel ...,joy,joy\n,1.047610,2.0,101.0,103.0,joy


In [11]:
y_pred = pred_df['prediction_post_processed']
y_true = pred_df['label']

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.595000
F1 score: 0.585054
Precision: 0.616562
Recall: 0.595000


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [12]:
# get average response time, vram usage and ram usage
request_time_avg = pred_df['request_time'].mean()
completion_tokens_avg = pred_df['completion_tokens'].mean()
prompt_tokens_avg = pred_df['prompt_tokens'].mean()
total_tokens_avg = pred_df['total_tokens'].mean()

print(f'Average response time: {request_time_avg}')
print(f'Average completion tokens: {completion_tokens_avg}')
print(f'Average prompt tokens: {prompt_tokens_avg}')
print(f'Average total tokens: {total_tokens_avg}')

Average response time: 1.0612834695577622
Average completion tokens: 2.4555
Average prompt tokens: 102.5175
Average total tokens: 104.973


In [13]:
input_token_price = 0.1/1_000_000
output_token_price = 0.4/1_000_000

# Calculate the cost of the requests
total_cost = 0
for index, row in pred_df.iterrows():
    completion_tokens = row['completion_tokens']
    prompt_tokens = row['prompt_tokens']
    cost  = completion_tokens * output_token_price + prompt_tokens * input_token_price
    total_cost += cost

print(f'Total cost: USD {total_cost}')

Total cost: USD 0.022467900000000027


In [14]:
with open('results/gemini_ZS_multiclass3.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {request_time_avg}\n')
    f.write(f'Average completion tokens: {completion_tokens_avg}\n')
    f.write(f'Average prompt tokens: {prompt_tokens_avg}\n')
    f.write(f'Average total tokens: {total_tokens_avg}\n')
    f.write(f'Total cost: USD {total_cost}\n')